How many Review-Commons-reviewed preprints have reviews/author replies without DOIs?

In [ ]:
import requests
import pandas

In [ ]:
dois_by_reviewing_service = requests.get("https://eeb.embo.org/api/v1/by_reviewing_service/").json()
revcom_data = [
    s
    for s in dois_by_reviewing_service
    if s["id"] == "review commons"
][0]
revcom_data

In [ ]:
pandas.DataFrame([(p["doi"], p["pub_date"]) for p in revcom_data["papers"]], columns=["doi", "published_at"])

In [ ]:
request_payload = {"dois": [p["doi"] for p in revcom_data["papers"]]}
data = requests.post("https://eeb.embo.org/api/v1/dois/", json=request_payload).json()
len(data), data[:3]

In [ ]:
reviews = [(a["doi"], r) for a in data for r in a["review_process"]["reviews"]]
responses = [(a["doi"], a["review_process"]["response"]) for a in data if a["review_process"]["response"] is not None]
len(reviews), len(responses)

In [ ]:
df = pandas.DataFrame(
    [
        (doi, r.get("doi", None), r["posting_date"])
        for doi, r in reviews + responses
    ],
    columns=["article_doi", "doi", "posting_date"]
)
df

In [ ]:
df[df["doi"].isna()]

In [ ]:
df[df["doi"].isna()]["article_doi"].value_counts()